In [1]:
import openai
import os

openai.__version__

'1.10.0'

In [2]:
miracl_n_hard_negs = 1000
miracl_n_recalls = [1,3,10,30,100,300,1000]
miracl_n_corpus = 2000
miracl_n_queries = 100
model_id = ""
dimension = -1
query_prefix = ""
passage_prefix = ""

In [3]:
# Parameters
model_id = "text-embedding-3-large"
dimension = 3072


In [ ]:
# cache
tmpdir = f"tmp/{model_id}_{dimension}"
os.makedirs(tmpdir, exist_ok=True)

# Model

In [4]:
import numpy as np
import dotenv
from langchain_openai import OpenAIEmbeddings

dotenv.load_dotenv("openai_key", override=True)

if "text-embedding-3" not in model_id:
    client = OpenAIEmbeddings(model=model_id)
else:
    client = OpenAIEmbeddings(model=model_id, dimensions=dimension)

def get_embeddings(texts: list[str]) -> np.ndarray:
    texts = [text.replace("\n", " ")[:2000] for text in texts]
    all_embeddings = []
    for i in range(0, len(texts), 500):
        embs = client.embed_documents(texts[i : i + 500])
        all_embeddings += embs
    return np.array(all_embeddings)


# Miracl
* Need access token for huggingface

In [13]:
import os
import dotenv
import json

dotenv.load_dotenv("huggingface_access_token", override=True)

True

In [14]:
import datasets

# query and positives
ds = datasets.load_dataset(
    "miracl/miracl", "ja", token=os.environ["HF_ACCESS_TOKEN"], split="dev"
)
ds

Found cached dataset miracl (G:/cache/miracl___miracl/ja/1.0.0/f598b4ee332f2b16e82c6c83ab1ba82e1a7777ef82e7ce3c1416f6b20a142313)


Dataset({
    features: ['query_id', 'query', 'positive_passages', 'negative_passages'],
    num_rows: 860
})

In [15]:
# all corpus texts
corpus = datasets.load_dataset("miracl/miracl-corpus", "ja")
corpus

Found cached dataset miracl-corpus (G:/cache/miracl___miracl-corpus/ja/1.0.0/16b566312c83a2e1f94d0813c8702b464b97f6b8959336adf062d289ce9b51fa)


  0%|          | 0/1 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['docid', 'title', 'text'],
        num_rows: 6953614
    })
})

In [16]:
# hard negatives
with open("./miracl_hard_negs_1000.json") as f:
    hn = json.loads(f.read())
len(hn), list(hn.keys())[:5], hn["0"].keys(), hn["0"]["docids"][:2], hn["0"]["indices"][
    :2
]

(860,
 ['0', '3', '4', '5', '7'],
 dict_keys(['docids', 'indices']),
 ['2681119#0', '2681119#1'],
 [1393435, 1393436])

In [17]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist


def get_text(corpus_item):
    return corpus_item["title"] + " " + corpus_item["text"]


corpus_dict = {item["docid"]: get_text(item) for item in corpus["train"]}

In [18]:
n_total_pos = 0
n_total_tp = 0

# only evaluate first 100 queries
for item in ds.select(range(100)):
    # query
    query_emb = get_embeddings([query_prefix + item["query"]])

    # passages are set(300 hard negatives + positives)
    positive_docids = [pp["docid"] for pp in item["positive_passages"]]
    positive_texts = [get_text(pp) for pp in item["positive_passages"]]
    hn_docids = hn[item["query_id"]]["docids"][:miracle_n_hard_negs]

    # drop hard negatives in positives
    hn_docids = [docid for docid in hn_docids if docid not in positive_docids]

    # search target
    target_docids = positive_docids + hn_docids
    target_texts = positive_texts + [corpus_dict[docid] for docid in hn_docids]

    # embedding
    tmppath = f'tmp/{model_id}_{dimension}_{item["query_id"]}.npy'
    if os.path.exists(tmppath):
        target_embs = np.load(tmppath)
    else:
        # use cache
        target_embs = get_embeddings([passage_prefix + text for text in target_texts])
        np.save(tmppath, target_embs)

    # topK
    topk_indices = np.argsort(cdist(query_emb, target_embs, metric="cosine"))[0][
        :miracle_n_recall
    ]

    n_pos = len(positive_docids)
    n_tp = len(
        set(topk_indices) & set(range(len(positive_docids)))
    )  # positives are first indices

    n_total_pos += n_pos
    n_total_tp += n_tp

miracl_recall = n_total_tp / n_total_pos

n_total_pos, n_total_tp, miracl_recall

(195, 164, 0.841025641025641)

# Output

In [19]:
model_id, dimension, jsts_score, jsick_score, miracl_recall

('text-embedding-3-large',
 3072,
 0.8376993902963759,
 0.8115338122021275,
 0.841025641025641)

In [20]:
import json

with open(f'./scores/{model_id.replace("/", "_")}_{dimension}.txt', "w") as f:
    f.write(
        json.dumps(
            {
                "model_id": model_id,
                "jsts": jsts_score,
                "jsick": jsick_score,
                "miracl": miracl_recall,
            }
        )
    )